# Implement K-Means Clustering in PyTorch — Solution

**Difficulty**: 🟢 Easy

**Companies**: Uber, LinkedIn, Google, Amazon, General FAANG

---

### Problem Statement

K-Means is one of the most widely used unsupervised learning algorithms. It partitions data into **K clusters** by iteratively assigning points to their nearest centroid and updating centroids to be the mean of their assigned points.

Your task is to implement K-Means clustering **from scratch** using only PyTorch tensor operations.

---

### Requirements

1. **Random Initialization** — Select K random data points as initial centroids.
2. **Assignment Step** — Assign each data point to the nearest centroid using Euclidean distance.
3. **Update Step** — Recompute each centroid as the mean of all points assigned to it.
4. **Convergence** — Stop when centroids no longer change (or change less than a tolerance).

---

### Constraints

- ✅ Use only PyTorch tensor operations (no sklearn, no Python loops over data points).
- ✅ Use `torch.cdist` or manual broadcasting for distance computation.
- ✅ Must work for any K and any dimensionality of data.
- ❌ Do **not** use any external clustering libraries.

---

<details>
  <summary>💡 Hint</summary>

  - Use `torch.cdist(data, centroids)` to compute pairwise distances efficiently. It returns a `(N, K)` distance matrix.
  - Use `torch.argmin(distances, dim=1)` to assign each point to its nearest centroid.
  - To update centroids, create a boolean mask for each cluster and compute `data[mask].mean(dim=0)`.
  - Check convergence by comparing old and new centroids with `torch.allclose()`.

</details>

---

In [ ]:
import torch

In [ ]:
# Generate synthetic data: 3 clusters in 2D
torch.manual_seed(42)

n_points_per_cluster = 100
true_centers = torch.tensor([[0.0, 0.0], [5.0, 5.0], [-5.0, 5.0]])

cluster_0 = torch.randn(n_points_per_cluster, 2) * 0.5 + true_centers[0]
cluster_1 = torch.randn(n_points_per_cluster, 2) * 0.5 + true_centers[1]
cluster_2 = torch.randn(n_points_per_cluster, 2) * 0.5 + true_centers[2]

data = torch.cat([cluster_0, cluster_1, cluster_2], dim=0)  # (300, 2)
print("Data shape:", data.shape)
print("True centers:\n", true_centers)

In [ ]:
def kmeans(data: torch.Tensor, k: int, max_iters: int = 100, tol: float = 1e-4):
    """
    K-Means clustering using PyTorch.

    Args:
        data (Tensor): Input data of shape (N, D).
        k (int): Number of clusters.
        max_iters (int): Maximum number of iterations.
        tol (float): Convergence tolerance for centroid movement.

    Returns:
        centroids (Tensor): Final centroids of shape (K, D).
        assignments (Tensor): Cluster assignment for each point, shape (N,).
    """
    N, D = data.shape

    # Step 1: Initialize centroids by randomly selecting K data points
    indices = torch.randperm(N)[:k]
    centroids = data[indices].clone()

    for i in range(max_iters):
        # Step 2: Compute distances from each point to each centroid
        distances = torch.cdist(data, centroids)  # (N, K)

        # Step 3: Assign each point to the nearest centroid
        assignments = torch.argmin(distances, dim=1)  # (N,)

        # Step 4: Update centroids as the mean of assigned points
        new_centroids = torch.zeros_like(centroids)
        for j in range(k):
            mask = assignments == j
            if mask.sum() > 0:
                new_centroids[j] = data[mask].mean(dim=0)
            else:
                # If no points assigned, keep the old centroid
                new_centroids[j] = centroids[j]

        # Step 5: Check for convergence
        if torch.allclose(centroids, new_centroids, atol=tol):
            print(f"Converged at iteration {i}")
            centroids = new_centroids
            break

        centroids = new_centroids

    return centroids, assignments

In [ ]:
# Validation
centroids, assignments = kmeans(data, k=3)
print("Learned centroids:\n", centroids)
print("True centers:\n", true_centers)

# Check that each learned centroid is close to one of the true centers
# (order may differ, so we match each learned centroid to its nearest true center)
dists = torch.cdist(centroids, true_centers)  # (3, 3)
min_dists = dists.min(dim=1).values
print("\nMin distance from each learned centroid to nearest true center:", min_dists)

assert (min_dists < 1.0).all(), f"Centroids too far from true centers: {min_dists}"
print("\nConvergence test PASSED!")

# Check that each cluster has roughly the right number of points
for i in range(3):
    count = (assignments == i).sum().item()
    print(f"Cluster {i}: {count} points")
    assert count > 50, f"Cluster {i} has too few points: {count}"

print("\nAll tests passed!")